# Joint Bayesian Operator Inference: Cubic Heat Equation

## Setup

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro
from numpyro.infer import autoguide

from core import (
    JaxCompatibleModel,
    compute_gp_derivatives,
    compute_derivatives_fourth_order,
    generate_rom_predictions,
    save_paper_figure,
    DataScaler,
    build_joint_bayesian_model,
    run_joint_svi,
    extract_gp_posterior,
    extract_derivative_posterior,
    gp_based_opinf_baseline,
    rbf_eval,
)
from config import (
    FullOrderModel, Basis,
    input_func_factory, input_parameters, test_parameters,
    time_domain, initial_conditions,
)
import step1_generate_data as step1
import opinf

numpyro.set_platform('cpu')
numpyro.set_host_device_count(4)

np.random.seed(42)
rng_key = jax.random.PRNGKey(42)

In [ ]:
# === EXPERIMENT CONFIGURATION ===
OPERATORS = "cAHBN"
IVP_METHOD = "BDF"
NUM_MODES = 5
NUM_TRAIN_ICS = 5

# Time domain
TRAINING_SPAN = (0, 1.0)
PREDICTION_SPAN = (0, 2.0)

# Data generation
NUM_SAMPLES = 65
NOISE_LEVEL = 0.02
NUM_EVAL_POINTS = 150  # Set to None for no densification

# Data scaling
USE_SCALED_DATA = False

# Inference settings
GUIDE = autoguide.AutoNormal
VERBOSE = True

# Hyperparameters
GAMMA = 1e0
GAMMA2 = 1e0

# GP hyperparameter priors (LogNormal)
GP_LENGTHSCALE_PRIOR_LOC = None
GP_LENGTHSCALE_PRIOR_SCALE = 1.0
GP_VARIANCE_PRIOR_LOC = None
GP_VARIANCE_PRIOR_SCALE = 0.5
GP_NOISE_PRIOR_LOC = None
GP_NOISE_PRIOR_SCALE = 1.0

# ODE constraint slack (γ₂) — hierarchical prior
LEARN_GAMMA2 = True           # If True, γ₂ is learned per mode; if False, fixed at GAMMA2
GAMMA2_PRIOR_LOC = None       # LogNormal loc for γ₂ prior; None defaults to log(GAMMA2)
GAMMA2_PRIOR_SCALE = 1.0      # LogNormal scale — larger = more freedom to find tightness

# SVI settings
NUM_SVI_STEPS = 50000
LEARNING_RATE = 1e-3
NUM_POSTERIOR_SAMPLES = 1000

# Plotting
NUM_REGRESSION_POINTS = 150

## 1. Generate Training Data

In [ ]:
train_params = input_parameters[:NUM_TRAIN_ICS]
test_params = test_parameters

print(f"Training parameters: {train_params}")
print(f"Test parameters: {test_params}")

sampler = step1.TrajectorySampler(
    TRAINING_SPAN, NUM_SAMPLES, NOISE_LEVEL, NUM_REGRESSION_POINTS, synced=False,
)

(
    all_true_states,
    all_time_sampled,
    all_snapshots,
    all_training_inputs,
) = sampler.multisample(train_params, plot=False)

time_domain_full = time_domain[
    (time_domain >= PREDICTION_SPAN[0]) & (time_domain <= PREDICTION_SPAN[1])
]

# Test trajectory
(
    snapshots_test,
    time_sampled_test,
    snapshots_test_noisy,
    inputs_test_train,
) = sampler.sample(test_params)

test_input_func = input_func_factory(test_params)
inputs_test = test_input_func(time_domain_full)

print(f"Training span: [{TRAINING_SPAN[0]}, {TRAINING_SPAN[1]}]")
print(f"Prediction span: [{PREDICTION_SPAN[0]}, {PREDICTION_SPAN[1]}]")

In [ ]:
# Stack all training data for basis fitting
snapshots_train = np.hstack(all_snapshots)

basis = Basis(num_vectors=NUM_MODES)
basis.fit(snapshots_train)

# Compress per-trajectory
all_snapshots_comp = [basis.compress(s) for s in all_snapshots]
all_true_compressed = [basis.compress(s) for s in all_true_states]
snapshots_test_compressed = basis.compress(snapshots_test)

# First trajectory for joint model fitting
snapshots_comp_first = all_snapshots_comp[0]
snapshots_first = all_snapshots[0]

print(f"Compressed shape (first traj): {snapshots_comp_first.shape}")
print(f"Cumulative energy: {basis.cumulative_energy:.4%}")

# Evaluation time domains
time_domain_eval_training = np.linspace(TRAINING_SPAN[0], TRAINING_SPAN[1], NUM_REGRESSION_POINTS)
time_domain_eval_prediction = np.linspace(PREDICTION_SPAN[0], PREDICTION_SPAN[1], NUM_REGRESSION_POINTS)

In [ ]:
if USE_SCALED_DATA:
    data_scaler = DataScaler(num_modes=NUM_MODES)
    data_scaler.fit(snapshots_comp_first)
    training_data = data_scaler.transform(snapshots_comp_first)
    print(f"Scaling enabled: {data_scaler}")
else:
    data_scaler = None
    training_data = snapshots_comp_first
    print("Scaling disabled: using raw POD coefficients")

## 2. Build ROM Structure

In [ ]:
first_input_func = input_func_factory(train_params[0])
inputs_first = np.array(first_input_func(all_time_sampled[0]))

rom = opinf.ROM(
    basis=basis,
    ddt_estimator=opinf.ddt.NonuniformFiniteDifferencer(all_time_sampled[0]),
    model=JaxCompatibleModel(operators=OPERATORS, solver=opinf.lstsq.L2Solver(regularizer=1e0))
)
rom.fit(states=snapshots_first, inputs=inputs_first)
print(f"Operator shape: {rom.model.operator_matrix.shape}")

## 3. Joint Bayesian Inference

In [ ]:
import time
start_time = time.time()

# Compute inputs at eval points for the first trajectory
if NUM_EVAL_POINTS is not None:
    time_eval_model = np.linspace(all_time_sampled[0][0], all_time_sampled[0][-1], NUM_EVAL_POINTS)
else:
    time_eval_model = all_time_sampled[0]
inputs_eval = np.array(first_input_func(time_eval_model))

joint_model, time_eval = build_joint_bayesian_model(
    rom=rom,
    num_modes=NUM_MODES,
    time_domain_sampled=all_time_sampled[0],
    snapshots=training_data,
    inputs_eval=inputs_eval,
    data_scaler=data_scaler if USE_SCALED_DATA else None,
    gp_lengthscale_prior_loc=GP_LENGTHSCALE_PRIOR_LOC,
    gp_lengthscale_prior_scale=GP_LENGTHSCALE_PRIOR_SCALE,
    gp_variance_prior_loc=GP_VARIANCE_PRIOR_LOC,
    gp_variance_prior_scale=GP_VARIANCE_PRIOR_SCALE,
    gp_noise_prior_loc=GP_NOISE_PRIOR_LOC,
    gp_noise_prior_scale=GP_NOISE_PRIOR_SCALE,
    gamma2_prior_loc=GAMMA2_PRIOR_LOC,
    gamma2_prior_scale=GAMMA2_PRIOR_SCALE,
    learn_gamma2=LEARN_GAMMA2,
    num_eval_points=NUM_EVAL_POINTS,
)
print(f"Joint model built. Eval points: {len(time_eval)}")

In [ ]:
svi_result = run_joint_svi(
    model=joint_model,
    rng_key=rng_key,
    gamma=GAMMA,
    gamma2=GAMMA2,
    num_steps=NUM_SVI_STEPS,
    learning_rate=LEARNING_RATE,
    num_samples=NUM_POSTERIOR_SAMPLES,
    verbose=VERBOSE,
    guide_class=GUIDE,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(svi_result.losses, lw=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('ELBO Loss')
ax.set_title('Joint SVI Convergence')
plt.tight_layout()
plt.show()

algo_time = time.time() - start_time
print(f"Algorithm runtime: {algo_time:.1f}s ({algo_time/60:.1f}min)")

In [ ]:
samples = svi_result.samples

print("Inferred GP Hyperparameters (median):")
for i in range(NUM_MODES):
    L_i = np.median(samples[f'lengthscale_{i}'])
    V_i = np.median(samples[f'variance_{i}'])
    N_i = np.median(samples[f'noise_{i}'])
    print(f"  Mode {i}: L={L_i:.4f}, V={V_i:.4f}, N={N_i:.6f}")
if LEARN_GAMMA2:
    print("\nInferred ODE constraint slack γ₂ (median):")
    for i in range(NUM_MODES):
        g2_i = np.median(samples[f'gamma2_{i}'])
        print(f"  Mode {i}: γ₂={g2_i:.4f}")


## 4. Diagnostic Plots

### Plot 1: GP Fit

In [ ]:
gp_means, gp_stds, Ls, Vs, Ns = extract_gp_posterior(
    samples, NUM_MODES, all_time_sampled[0], time_domain_eval_training, training_data,
)

fig, axes = plt.subplots(2, NUM_MODES, figsize=(4 * NUM_MODES, 6), sharex=True)
if NUM_MODES == 1:
    axes = axes[:, np.newaxis]

for i in range(NUM_MODES):
    ax = axes[0, i]
    ax.plot(all_time_sampled[0], training_data[i], 'k.', ms=3, alpha=0.4, label='Data')
    ax.plot(time_domain_eval_training, gp_means[i], 'b-', lw=2, label='GP mean')
    ax.fill_between(
        time_domain_eval_training,
        gp_means[i] - 2 * gp_stds[i],
        gp_means[i] + 2 * gp_stds[i],
        alpha=0.2, color='blue', label='\u00b12\u03c3',
    )
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('State')
        ax.legend(fontsize=7)

    ax2 = axes[1, i]
    ax2.hist(np.array(samples[f'lengthscale_{i}']), bins=30, alpha=0.5, label=f'\u2113={Ls[i]:.3f}', density=True)
    ax2.hist(np.array(samples[f'variance_{i}']), bins=30, alpha=0.5, label=f'\u03c3\u00b2={Vs[i]:.3f}', density=True)
    ax2.legend(fontsize=7)
    if i == 0:
        ax2.set_ylabel('Density')
    ax2.set_xlabel('Time')

fig.suptitle('Plot 1: GP Fit with Bayesian Hyperparameters', fontsize=14)
plt.tight_layout()
plt.show()

### Plot 2: Derivative GP Fit

In [ ]:
mu_z, std_z = extract_derivative_posterior(
    Ls, Vs, Ns, all_time_sampled[0], time_domain_eval_training, training_data,
)

fd_derivs = compute_derivatives_fourth_order(training_data, all_time_sampled[0])

fig, axes = plt.subplots(1, NUM_MODES, figsize=(4 * NUM_MODES, 3.5), sharex=True)
if NUM_MODES == 1:
    axes = [axes]

for i in range(NUM_MODES):
    ax = axes[i]
    ax.plot(all_time_sampled[0], fd_derivs[i], 'k.', ms=3, alpha=0.4, label='FD deriv')
    ax.plot(time_domain_eval_training, mu_z[i], 'r-', lw=2, label='GP deriv mean')
    ax.fill_between(
        time_domain_eval_training,
        mu_z[i] - 2 * std_z[i],
        mu_z[i] + 2 * std_z[i],
        alpha=0.2, color='red', label='\u00b12\u03c3',
    )
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('dq/dt')
        ax.legend(fontsize=7)
    ax.set_xlabel('Time')

fig.suptitle('Plot 2: Derivative GP Fit', fontsize=14)
plt.tight_layout()
plt.show()

### Plot 3: GP-based OpInf Integration (Baseline)

In [ ]:
gp_means_train, _, _, _, _ = extract_gp_posterior(
    samples, NUM_MODES, all_time_sampled[0], all_time_sampled[0], training_data,
)

gp_baseline_result = gp_based_opinf_baseline(
    basis=basis,
    gp_means=gp_means_train,
    time_eval=all_time_sampled[0],
    snapshots_compressed=snapshots_comp_first,
    operators=OPERATORS,
    inputs=inputs_first,
    input_func=first_input_func,
    ivp_method=IVP_METHOD,
)
print(f"GP baseline reg: {gp_baseline_result.best_reg:.2e}, error: {gp_baseline_result.best_error:.4%}")

gp_baseline_rom = gp_baseline_result.rom
gp_baseline_rom.model._extract_operators(np.array(gp_baseline_result.operator))
try:
    gp_pred = gp_baseline_rom.model.predict(
        state0=snapshots_comp_first[:, 0],
        t=time_domain_eval_prediction,
        input_func=first_input_func,
        method=IVP_METHOD,
    )
    gp_pred_sol = gp_baseline_rom.model.predict_result_
    gp_baseline_stable = gp_pred_sol.y.shape[1] >= len(time_domain_eval_prediction)
except Exception:
    gp_baseline_stable = False

fig, axes = plt.subplots(1, NUM_MODES, figsize=(4 * NUM_MODES, 3.5), sharex=True)
if NUM_MODES == 1:
    axes = [axes]

for i in range(NUM_MODES):
    ax = axes[i]
    ax.plot(time_domain_full, all_true_compressed[0][i], 'k-', lw=1.5, label='Truth')
    ax.plot(all_time_sampled[0], snapshots_comp_first[i], 'k.', ms=3, alpha=0.4, label='Data')
    if gp_baseline_stable:
        ax.plot(time_domain_eval_prediction, gp_pred_sol.y[i], 'g--', lw=2, label='GP OpInf')
    ax.axvline(TRAINING_SPAN[1], color='gray', ls=':', lw=1, alpha=0.5)
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('State')
        ax.legend(fontsize=7)
    ax.set_xlabel('Time')

fig.suptitle('Plot 3: Integration of GP-based OpInf (Best Case Baseline)', fontsize=14)
plt.tight_layout()
plt.show()

### Plot 4: Latent State Fit

In [ ]:
fig, axes = plt.subplots(1, NUM_MODES, figsize=(4 * NUM_MODES, 3.5), sharex=True)
if NUM_MODES == 1:
    axes = [axes]

for i in range(NUM_MODES):
    ax = axes[i]
    X_i = np.array(samples[f'X_{i}'])
    X_median = np.median(X_i, axis=0)
    X_q05 = np.percentile(X_i, 5, axis=0)
    X_q95 = np.percentile(X_i, 95, axis=0)

    ax.plot(all_time_sampled[0], training_data[i], 'k.', ms=3, alpha=0.4, label='Data')
    ax.plot(all_time_sampled[0], X_median, 'b-', lw=2, label='X median')
    ax.fill_between(all_time_sampled[0], X_q05, X_q95, alpha=0.2, color='blue', label='5-95%')
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('State')
        ax.legend(fontsize=7)
    ax.set_xlabel('Time')

fig.suptitle('Plot 4: Latent State Fit (X)', fontsize=14)
plt.tight_layout()
plt.show()

### Plot 5: Derivative Operator Fit

In [ ]:
from core.bayesian_opinf import _find_operator_samples

O_samples = _find_operator_samples(samples, "O")
if O_samples.ndim == 2:
    O_samples = O_samples[np.newaxis, ...]

O_median = np.median(O_samples, axis=0)

if f'X_eval_0' in samples:
    Xs_eval_median = np.array([np.median(samples[f'X_eval_{i}'], axis=0) for i in range(NUM_MODES)])
else:
    Xs_eval_median = np.array([np.median(samples[f'X_{i}'], axis=0) for i in range(NUM_MODES)])

if USE_SCALED_DATA and data_scaler is not None:
    Xs_eval_orig = np.array([
        Xs_eval_median[i] * data_scaler.stds_[i, 0] + data_scaler.means_[i, 0]
        for i in range(NUM_MODES)
    ])
else:
    Xs_eval_orig = Xs_eval_median

# Need inputs at eval points
t_eval_plot = time_eval if f'X_eval_0' in samples else all_time_sampled[0]
inputs_at_eval = np.array(first_input_func(t_eval_plot))

f_X = np.array(rom.model._assemble_data_matrix(
    jnp.array(Xs_eval_orig), inputs=jnp.array(inputs_at_eval)
) @ jnp.array(O_median).T)

if USE_SCALED_DATA and data_scaler is not None:
    f_X_scaled = np.array([f_X.T[i] / data_scaler.stds_[i, 0] for i in range(NUM_MODES)])
else:
    f_X_scaled = f_X.T

fig, axes = plt.subplots(1, NUM_MODES, figsize=(4 * NUM_MODES, 3.5), sharex=True)
if NUM_MODES == 1:
    axes = [axes]

for i in range(NUM_MODES):
    ax = axes[i]
    ax.plot(time_domain_eval_training, mu_z[i], 'r-', lw=1.5, label='GP deriv \u03bc_z')
    ax.plot(t_eval_plot, f_X_scaled[i], 'b--', lw=2, label='f(X)O^T')
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('dq/dt')
        ax.legend(fontsize=7)
    ax.set_xlabel('Time')

fig.suptitle('Plot 5: Derivative Operator Fit (f(X)O^T vs GP derivatives)', fontsize=14)
plt.tight_layout()
plt.show()

### Plot 6: Integrated Operator Fit (ROM Predictions)

In [ ]:
Os, Xs, rom_solves = generate_rom_predictions(
    samples=samples, rom=rom,
    snapshots_compressed=snapshots_comp_first,
    time_eval=time_domain_eval_prediction,
    num_modes=NUM_MODES, num_pulls=200,
    input_func=first_input_func,
    data_scaler=data_scaler if USE_SCALED_DATA else None,
    ivp_method=IVP_METHOD,
)
print(f"Operator samples: {len(Os)}, Stable solves: {len(rom_solves)}")

fig, axes = plt.subplots(1, NUM_MODES, figsize=(4 * NUM_MODES, 3.5), sharex=True)
if NUM_MODES == 1:
    axes = [axes]

for i in range(NUM_MODES):
    ax = axes[i]
    ax.plot(time_domain_full, all_true_compressed[0][i], 'k-', lw=1.5, label='Truth')
    ax.plot(all_time_sampled[0], snapshots_comp_first[i], 'k.', ms=3, alpha=0.4, label='Data')

    if len(rom_solves) > 0:
        rom_arr = np.array(rom_solves)
        median = np.median(rom_arr[:, i, :], axis=0)
        q05 = np.percentile(rom_arr[:, i, :], 5, axis=0)
        q95 = np.percentile(rom_arr[:, i, :], 95, axis=0)
        ax.plot(time_domain_eval_prediction, median, 'purple', ls='--', lw=2, label='ROM median')
        ax.fill_between(time_domain_eval_prediction, q05, q95, alpha=0.2, color='purple', label='5-95%')

    ax.axvline(TRAINING_SPAN[1], color='gray', ls=':', lw=1, alpha=0.5)
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('State')
        ax.legend(fontsize=7)
    ax.set_xlabel('Time')

fig.suptitle('Plot 6: Integrated Operator Fit (ROM Predictions with UQ)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT SUMMARY: Cubic Heat (Joint Bayesian)")
print("=" * 60)
print(f"Operators: {OPERATORS}")
print(f"Modes: {NUM_MODES}")
print(f"Training ICs: {NUM_TRAIN_ICS} (fitted on first IC)")
print(f"Training span: [{TRAINING_SPAN[0]}, {TRAINING_SPAN[1]}]")
print(f"Prediction span: [{PREDICTION_SPAN[0]}, {PREDICTION_SPAN[1]}]")
print(f"Scaling: {'enabled' if USE_SCALED_DATA else 'disabled'}")
print(f"Gamma (operator): {GAMMA}")
print(f"Gamma2 (ODE): {GAMMA2}")
print(f"Guide: {GUIDE.__name__}")
print(f"SVI steps: {NUM_SVI_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"\nInferred GP Hyperparameters:")
for i in range(NUM_MODES):
    print(f"  Mode {i}: L={Ls[i]:.4f}, V={Vs[i]:.4f}, N={Ns[i]:.6f}")
print(f"\nOperator samples: {len(Os)}")
print(f"Stable ROM solves: {len(rom_solves)}")
print(f"Algorithm runtime: {algo_time:.1f}s ({algo_time/60:.1f}min)")